# 03 · Tabelas gerenciadas × não gerenciadas (Delta)

- **Não gerenciada (externa)**: criada com `LOCATION`. O Spark guarda só os metadados; os dados ficam no caminho informado (aqui, o MinIO). `DROP TABLE` **não apaga os dados**.
- **Gerenciada**: criada sem `LOCATION`. O Spark decide onde guardar (`spark.sql.warehouse.dir`). `DROP TABLE` **apaga metadados e dados**.

No trabalho 1 as tabelas `carros` foram criadas com `LOCATION` (não gerenciadas).

In [1]:
import shutil
from pathlib import Path

from spark_delta_minio_sqlserver.spark_session import create_session

spark = create_session("03-gerenciadas")
spark.sparkContext.setLogLevel("ERROR")

spark.sql("DROP TABLE IF EXISTS carro_gerenciada")
spark.sql("DROP TABLE IF EXISTS carro_externa")
shutil.rmtree("spark-warehouse/carro_gerenciada", ignore_errors=True)

26/10/11 12:45:34 WARN Utils: Your hostname, NF-ENG-ISABELLEFELTRIN resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/11 12:45:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/isafel/spark-delta-minio-sqlserver/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/isafel/.ivy2/cache
The jars for the packages stored in: /home/isafel/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
com.microsoft.sqlserver#mssql-jdbc added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-9dfdd6ce-bc67-4916-a516-3b116291d1ba;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.2 in central
	found io.delta#delta-storage;3.3.2 in central
	found org.antlr#antlr4-runtime;4.9.3 in central


	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found com.microsoft.sqlserver#mssql-jdbc;12.8.1.jre11 in central
:: resolution report :: resolve 286ms :: artifacts dl 12ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	com.microsoft.sqlserver#mssql-jdbc;12.8.1.jre11 from central in [default]
	io.delta#delta-spark_2.12;3.3.2 from central in [default]
	io.delta#delta-storage;3.3.2 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	------

26/10/11 12:45:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


## Não gerenciada: aponta para a tabela do `bronze`

In [2]:
spark.sql("""
    CREATE TABLE carro_externa USING delta
    LOCATION 's3a://bronze/carro'
""")
spark.sql("SELECT COUNT(*) AS linhas FROM carro_externa").show()

+------+
|linhas|
+------+
|     8|
+------+



## Gerenciada: cópia criada sem `LOCATION`

In [3]:
spark.sql("""
    CREATE TABLE carro_gerenciada USING delta
    AS SELECT * FROM carro_externa
""")
spark.sql("SELECT COUNT(*) AS linhas FROM carro_gerenciada").show()

+------+
|linhas|
+------+
|     8|
+------+



## Onde cada uma guarda os dados?

In [4]:
for tabela in ["carro_externa", "carro_gerenciada"]:
    print(tabela)
    (
        spark.sql(f"DESCRIBE EXTENDED {tabela}")
        .filter("col_name IN ('Type', 'Location')")
        .show(truncate=False)
    )

carro_externa


+--------+------------------+-------+
|col_name|data_type         |comment|
+--------+------------------+-------+
|Type    |EXTERNAL          |       |
|Location|s3a://bronze/carro|       |
+--------+------------------+-------+

carro_gerenciada
+--------+----------------------------------------------------------------------------------------+-------+
|col_name|data_type                                                                               |comment|
+--------+----------------------------------------------------------------------------------------+-------+
|Type    |MANAGED                                                                                 |       |
|Location|file:/home/isafel/spark-delta-minio-sqlserver/notebooks/spark-warehouse/carro_gerenciada|       |
+--------+----------------------------------------------------------------------------------------+-------+



## `DROP TABLE` em cada uma

In [5]:
spark.sql("DROP TABLE carro_externa")
spark.sql("DROP TABLE carro_gerenciada")

print("Pasta da gerenciada ainda existe?", Path("spark-warehouse/carro_gerenciada").exists())
print("Linhas da tabela externa no MinIO (os dados continuam lá):")
spark.read.format("delta").load("s3a://bronze/carro").count()

Pasta da gerenciada ainda existe? False
Linhas da tabela externa no MinIO (os dados continuam lá):


8

| | Não gerenciada (externa) | Gerenciada |
|---|---|---|
| Criação | `... LOCATION '<caminho>'` | sem `LOCATION` |
| Dados ficam | no caminho informado (MinIO) | em `spark.sql.warehouse.dir` |
| `DROP TABLE` | remove só o metadado | remove metadado **e** dados |
| Uso típico | camadas do data lake (landing, bronze), dados compartilhados | tabelas temporárias/intermediárias |